# **02_McNemar_test.ipynb**

**Programmers:**
* Albonia, Jade Lorenz M.
* Caspe, Mark Vincent G.
* Rivera, Rei Djemf M.
* Velante, Kamilah Kaye M.
* Villegas, Jedidiah S.


**Date Written:** September 2025

**Date Revised:** December 2025

---

### **System Context**
This notebook executes the **Post-Hoc Statistical Analysis** for the A-EYE project. Following the generation of classification results in Phase 2, this module determines if the performance difference between the **Baseline (MobileViT)** and the **A-EYE (4-Ring)** model is statistically significant or attributable to random chance. It serves as the mathematical validation for **Chapter 4: Results and Discussion**.

### **Purpose**
To rigorously validate the comparative results using **McNemar's Test**. Specifically, it automates:
1.  **Contingency Table Construction:** Building the $2 \times 2$ matrix of paired agreements and disagreements.
2.  **Hypothesis Testing:** Calculating the $\chi^2$ statistic and $p$-value to reject (or fail to reject) the Null Hypothesis ($H_0$: The models perform identically).
3.  **Granular Error Analysis:** Generating a side-by-side CSV report (`final_detailed_predictions.csv`) that isolates exactly which images caused the disagreement.

---

### **Technical Architecture (Data Structures & Algorithms)**

**1. Data Structures**
* **Contingency Matrix ($2 \times 2$ Array):** A dense matrix storing the counts of paired outcomes:
    * $N_{00}$: Both models correct (Concordant)
    * $N_{11}$: Both models wrong (Concordant)
    * $N_{01}$: Baseline correct, A-EYE wrong (Discordant)
    * $N_{10}$: Baseline wrong, A-EYE correct (Discordant)
* **Comparison DataFrame:** A Pandas structure that aggregates the filename, Ground Truth, and predictions from both models for row-by-row analysis.

**2. Algorithms**
* **McNemar's Test ($\chi^2$):** A non-parametric paired test used on nominal data. It focuses exclusively on the **discordant pairs** ($N_{01}$ and $N_{10}$) to measure marginal homogeneity.

* **Binary Thresholding:** Converts continuous probability outputs (Sigmoid) into rigid binary classes (Mature/Immature) using a standard $\theta = 0.5$ threshold before comparison.

**3. Control Flow**
* **Data Alignment:** Loads two separate model state dictionaries and forces them to predict on the *exact same* sequence of test images to ensure valid pairing.
* **Statistical Computation:** Uses `statsmodels.stats.contingency_tables.mcnemar` to compute the P-value.
* **Reporting:** Conditionals categorize every test sample into "Both Correct," "Baseline Wins," or "A-EYE Wins" and exports this logic to CSV for manual review.

---

## Step 1: Setup Environment

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

REPO_URL = "https://github.com/its-levi0sa/a-eye-cataract-maturity-classification-tool.git"
PROJECT_DIR = "A-EYE"

# --- Clone or Pull Latest Code ---
if os.path.exists(PROJECT_DIR):
    print("Repository already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull
else:
    print("Cloning repository...")
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

# --- Configure Paths ---
if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

!pip install -q -r requirements.txt

print("\n✅ Environment setup complete.")

## Step 2: Define Ensemble Prediction Function

In [ ]:
import torch
import glob
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from src.aeye_model import AEyeModel
from src.baseline_model import mobilevit_s
from src.data_utils import get_transforms, AlbumentationsDataset
import torch.nn as nn

def get_ensemble_predictions(model_type, model_dir, data_dir, num_rings=4, dims=[32, 64, 128, 160], embed_dim=256):
    """
    Runs ensemble inference and returns a list of predicted labels (0 or 1).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Find Models
    model_paths = sorted(glob.glob(os.path.join(model_dir, '*.pth')))
    if not model_paths:
        print(f"❌ Error: No .pth files found in {model_dir}")
        return None

    print(f"\n--- Loading {len(model_paths)} models for {model_type.upper()} ---")

    # 2. Load Ensemble
    models = []
    for path in model_paths:
        if model_type == 'baseline':
            model = mobilevit_s()
            model.fc = nn.Linear(model.fc.in_features, 1)
        elif model_type == 'aeye':
            config = {'dims': dims, 'embed_dim': embed_dim, 'num_rings': num_rings}
            model = AEyeModel(config)

        model.load_state_dict(torch.load(path, map_location=device))
        model.to(device).eval()
        models.append(model)

    # 3. Load Data (Sequential, No Shuffle)
    image_paths = sorted(glob.glob(os.path.join(data_dir, '*/*.[jp][pn]g')))
    labels = [0 if 'immature' in path else 1 for path in image_paths]

    dataset = AlbumentationsDataset(image_paths, labels, transform=get_transforms(is_train=False))
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

    # 4. Inference
    all_preds = []

    with torch.no_grad():
        for images, _ in tqdm(loader, desc=f"Evaluating {model_type.upper()}"):
            images = images.to(device)

            # Get predictions from all 5 models
            batch_outputs = []
            for model in models:
                output = torch.sigmoid(model(images)).cpu().numpy().flatten()
                batch_outputs.append(output)

            # Ensemble Averaging
            avg_output = np.mean(batch_outputs, axis=0)
            rounded_preds = np.round(avg_output).astype(int)
            all_preds.extend(rounded_preds)

    return all_preds, image_paths, labels

print("✅ Prediction function ready.")

## Step 3: Generate and Compile Report

In [ ]:
# --- CONFIGURATION ---
BASELINE_DIR = "saved_models/baseline"
AEYE_DIR = "saved_models/aeye_4_ring"
TEST_DATA_DIR = "data/test"

# 1. Run Inference
base_preds, file_paths, true_labels = get_ensemble_predictions('baseline', BASELINE_DIR, TEST_DATA_DIR)
aeye_preds, _, _ = get_ensemble_predictions('aeye', AEYE_DIR, TEST_DATA_DIR, num_rings=4)

# 2. Create DataFrame
data = []
for i in range(len(file_paths)):
    filename = os.path.basename(file_paths[i])
    truth = "Mature" if true_labels[i] == 1 else "Immature"
    p_base = "Mature" if base_preds[i] == 1 else "Immature"
    p_aeye = "Mature" if aeye_preds[i] == 1 else "Immature"

    # Determine Status
    if p_base == truth and p_aeye == truth:
        status = "Both Correct"
    elif p_base != truth and p_aeye != truth:
        status = "Both Wrong"
    elif p_aeye == truth:
        status = "✅ A-EYE Wins"
    else:
        status = "❌ Baseline Wins"

    data.append([filename, truth, p_base, p_aeye, status])

df = pd.DataFrame(data, columns=["Image", "Doctor's Label", "Baseline", "A-EYE", "Comparison"])

# 3. Display and Save
pd.set_option('display.max_rows', None)
print("\n--- DETAILED PREDICTION REPORT ---")
display(df)

# Save to CSV
df.to_csv("results/final_detailed_predictions.csv", index=False)
print("\n✅ Report saved to 'results/final_detailed_predictions.csv'")

# 4. Summary of Disagreements
disagreements = df[df['Comparison'].str.contains('Wins')]
print(f"\n--- Analysis of Disagreements ({len(disagreements)} images) ---")
display(disagreements)